# MP2 Part 3.1: Commits Before and After the Longest Gap (lwang111)

For each project, takes the last 10 commits before its longest inactivity gap and the first 10 commits after it (or all if fewer exist), saves them to `lwang111_project_gap_commits.csv`, and records the top two commit themes on each side in `lwang111_project_stats.csv`.

Run `lwang111_vis.ipynb` first, because this notebook reads the gap columns it writes.

In [1]:
import pandas as pd

NETID = "lwang111"
COMMITS_PER_SIDE = 10

## Load commits and gap boundaries

In [2]:
commits = pd.read_csv(f"{NETID}_project_summary.csv", sep=";")
commit_datetimes = pd.to_datetime(commits["time"], unit="s")
commits["Month"] = commit_datetimes.dt.strftime("%Y-%m")

project_stats = pd.read_csv(f"{NETID}_project_stats.csv", sep=";")
gap_bounds = project_stats.set_index("project")[["LongestGapStart (YYYY-MM)", "LongestGapEnd (YYYY-MM)"]]
gap_bounds

,LongestGapStart (YYYY-MM),LongestGapEnd (YYYY-MM)
project,,
asgr_profound,2024-03,2024-08
homalg-project_toricvarieties_project,2023-01,2024-11
lanl_elements,2024-01,2024-06
ldbc_sigmod2014-pc,2022-08,2025-04
libatoms_quip,2024-10,2025-03
nanocomp_meep,2012-12,2014-02
opticspy_lightpipes,2016-01,2016-12
ph-rast_bmgarch,2022-01,2023-08
wiris_moodle-filter_wiris,2022-08,2022-09


## Collect commits around each gap

The gap's start and end months have zero commits, so "before" means any month earlier than the gap start and "after" means any month later than the gap end.

In [3]:
# Return the last N commits before and the first N commits after a project's longest gap.
def select_commits_around_gap(project_commits, gap_start_month, gap_end_month):
    ordered_commits = project_commits.sort_values("time")

    is_before_gap = ordered_commits["Month"] < gap_start_month
    is_after_gap = ordered_commits["Month"] > gap_end_month
    before_gap = ordered_commits[is_before_gap].tail(COMMITS_PER_SIDE)
    after_gap = ordered_commits[is_after_gap].head(COMMITS_PER_SIDE)

    return before_gap, after_gap


summary_columns = ["project_wocid", "commit_sha1", "author", "time", "commit message"]
gap_commit_frames = []

for project, bounds in gap_bounds.iterrows():
    project_commits = commits[commits["project_wocid"] == project]
    before_gap, after_gap = select_commits_around_gap(
        project_commits,
        bounds["LongestGapStart (YYYY-MM)"],
        bounds["LongestGapEnd (YYYY-MM)"],
    )

    for label, side_commits in [("pre", before_gap), ("post", after_gap)]:
        labeled = side_commits[summary_columns].copy()
        labeled.insert(0, "pre/post", label)
        gap_commit_frames.append(labeled)

gap_commits = pd.concat(gap_commit_frames, ignore_index=True)
gap_commits.to_csv(f"{NETID}_project_gap_commits.csv", sep=";", index=False)

gap_commits.groupby(["project_wocid", "pre/post"], sort=False).size().unstack()

pre/post,pre,post
project_wocid,,
asgr_profound,10,10
homalg-project_toricvarieties_project,10,7
lanl_elements,10,10
ldbc_sigmod2014-pc,10,1
libatoms_quip,10,10
nanocomp_meep,10,10
opticspy_lightpipes,10,10
ph-rast_bmgarch,10,10
wiris_moodle-filter_wiris,10,10


## Read the commit messages

Printed per project so the theme labels below can be checked against the evidence. Some messages appear twice with different SHAs; these are separate commits (e.g. the same change committed on two branches), so both are kept.

In [4]:
# Print a short one-line view of each commit around a project's gap.
def print_gap_commits(project):
    project_gap_commits = gap_commits[gap_commits["project_wocid"] == project]
    print(f"===== {project}")

    for _, row in project_gap_commits.iterrows():
        commit_month = pd.to_datetime(row["time"], unit="s").strftime("%Y-%m")
        author_name = row["author"].split("<")[0].strip()
        first_message_line = str(row["commit message"]).strip().splitlines()[0][:110]
        print(f"{row['pre/post']:<4} {commit_month}  {author_name[:22]:<22} {first_message_line}")
    print()


for project in gap_bounds.index:
    print_gap_commits(project)

===== asgr_profound
pre  2023-08  Aaron Robotham         Oops
pre  2023-09  Aaron Robotham         Some check bugs fixed.
pre  2023-10  Aaron Robotham         Some small updated to improve profound object inputs.
pre  2023-11  Aaron Robotham         Deprecated functions finally removed.
pre  2023-11  Aaron Robotham         Small updates.
pre  2023-11  Aaron Robotham         Better mask drawing.
pre  2023-11  Aaron Robotham         Rfits only support.
pre  2024-02  Aaron Robotham         Doc fix
pre  2024-02  Aaron Robotham         Merge pull request #25 from asgr/keyvalues_only
pre  2024-02  Aaron Robotham         profoundProFound will now load a Rfits_pointer object. This means profoundMultiBand can also be (e.g.) a list 
post 2024-09  Aaron Robotham         app_diam default is now NA (to stop silly mistakes...)
post 2024-09  Aaron Robotham         Work in progress. Need ability to pass in segim_orig and segim so both colour and total photometry modes work.
post 2024-10  Aaron Robotha

## Theme classification

Each commit was labeled with one of: Feature development, Bug fixes, Documentation updates, Dependency updates, Release, Security patches, Automated bot contributions, Other. The two most frequent labels on each side are recorded. "Other" covers refactoring, CI configuration, build/asset compilation, mailmap edits, and auto-generated `Merge <sha> into <sha>` PR-ref commits. When a side only shows one theme, the second slot is `N/A`.

| Project | Before gap | After gap |
|---|---|---|
| asgr_profound | **Feature development** (Rfits/keyvalues support, mask drawing, Rfits_pointer loading) and **Bug fixes** ("Some check bugs fixed", "Oops") | **Feature development** (segim_orig option, profoundMultiBand updates) and **Bug fixes** (segimlist fix, WAVES corner case, app_diam default) |
| homalg-project_toricvarieties_project | **Other**: refactoring (removing old methods, .gitignore, dropping unused arguments), then **Bug fixes** (lower-bound check) | **Documentation updates** (4× "Update Readme.md", maintainer list) and **Other** (avoid `InstallValue` for newer GAP), with only 7 commits after the gap |
| lanl_elements | **Other** (code cleanup, merges from master, auto-generated merge refs) and **Dependency updates** (MATAR submodule updates) | **Dependency updates**: nearly every commit bumps the MATAR submodule (for updated Kokkos); **Other** for auto merge refs |
| ldbc_sigmod2014-pc | **Feature development** (adding logging to queries Q1–Q4) and **Documentation updates** (LICENSE/NOTICE, DOI) | **Bug fixes** ("Fix missing import"), the only commit after the gap |
| libatoms_quip | **Dependency updates** (f90wrap version bound, GAP version update) and **Feature development** (ARM macOS arch file, SphericalIterative functions) | **Other**: all 10 are one refactor that renames F95 sources/references to F90 and removes a test program |
| nanocomp_meep | **Bug fixes** (NaNs in LDOS, single-precision build, LAPACK linking, gcc workaround) and **Release** (version bump to 1.2.1, release date, copyright update) | **Other** (SVN→git conversion, mailmap, removing obsolete TODOs) and **Feature development** (absorber PML type, public debugging methods) |
| opticspy_lightpipes | **Documentation updates** (README, command reference, examples) and **Release** ("add newest version") | **Feature development** (FFTW3-accelerated builds for Python 2.7/3.5, adding sources) and **Documentation updates** (4× README) |
| ph-rast_bmgarch | **Release**: CRAN submission prep (rhub checks, cran-comments, local check), then **Documentation updates** (spelling, URLs, NEWS) | **Other**: GitHub Actions/CI overhaul (setup-r v2, workflow YAML, Ubuntu 22.04), then **Dependency updates** (rstan array syntax, roxygen) |
| wiris_moodle-filter_wiris | **Bug fixes** (line endings, non-printable characters, rendertype default) and **Other** (CI matrix for Moodle 4) | **Bug fixes** (flaky Moodle 4.1 tests and their revert) and **Release** (publish 8.0.0 and 8.0.1, changelog) |
| wurmlab_sequenceserver | **Feature development** (print layout, fold graph by default) and **Bug fixes** (disabled-button fix, failing spec fix) | **Feature development** (default collapse state, collapse-preference refactor) and **Other** (compiling front-end assets) |

In [5]:
BEFORE_THEMES = {
    "asgr_profound": "Feature development:Bug fixes",
    "homalg-project_toricvarieties_project": "Other:Bug fixes",
    "lanl_elements": "Other:Dependency updates",
    "ldbc_sigmod2014-pc": "Feature development:Documentation updates",
    "libatoms_quip": "Dependency updates:Feature development",
    "nanocomp_meep": "Bug fixes:Release",
    "opticspy_lightpipes": "Documentation updates:Release",
    "ph-rast_bmgarch": "Release:Documentation updates",
    "wiris_moodle-filter_wiris": "Bug fixes:Other",
    "wurmlab_sequenceserver": "Feature development:Bug fixes",
}

AFTER_THEMES = {
    "asgr_profound": "Feature development:Bug fixes",
    "homalg-project_toricvarieties_project": "Documentation updates:Other",
    "lanl_elements": "Dependency updates:Other",
    "ldbc_sigmod2014-pc": "Bug fixes:N/A",
    "libatoms_quip": "Other:N/A",
    "nanocomp_meep": "Other:Feature development",
    "opticspy_lightpipes": "Feature development:Documentation updates",
    "ph-rast_bmgarch": "Other:Dependency updates",
    "wiris_moodle-filter_wiris": "Bug fixes:Release",
    "wurmlab_sequenceserver": "Feature development:Other",
}

## Add themes to lwang111_project_stats.csv

In [6]:
# Re-running replaces earlier theme columns instead of duplicating them.
theme_columns = ["BeforeThemes", "AfterThemes"]
project_stats = project_stats.drop(columns=theme_columns, errors="ignore")

project_stats["BeforeThemes"] = project_stats["project"].map(BEFORE_THEMES)
project_stats["AfterThemes"] = project_stats["project"].map(AFTER_THEMES)

project_stats.to_csv(f"{NETID}_project_stats.csv", sep=";", index=False)
project_stats[["project", "BeforeThemes", "AfterThemes"]]

,project,BeforeThemes,AfterThemes
0,asgr_profound,Feature development:Bug fixes,Feature development:Bug fixes
1,homalg-project_toricvarieties_project,Other:Bug fixes,Documentation updates:Other
2,lanl_elements,Other:Dependency updates,Dependency updates:Other
3,ldbc_sigmod2014-pc,Feature development:Documentation updates,Bug fixes:N/A
4,libatoms_quip,Dependency updates:Feature development,Other:N/A
5,nanocomp_meep,Bug fixes:Release,Other:Feature development
6,opticspy_lightpipes,Documentation updates:Release,Feature development:Documentation updates
7,ph-rast_bmgarch,Release:Documentation updates,Other:Dependency updates
8,wiris_moodle-filter_wiris,Bug fixes:Other,Bug fixes:Release
9,wurmlab_sequenceserver,Feature development:Bug fixes,Feature development:Other
